# Demo 8 · Hull Form Variation from a Parent: Lackenby and Stretch
**Ship and Ocean Systems Design 2026 · optional, not graded · about 1 hour**

**Objectives**
1. Load a parent hull as a table of offsets and check its hydrostatics against the published values.
2. Change its prismatic coefficient and LCB with the Lackenby section shift.
3. Stretch the result to new L, B and T, and see which coefficients survive.
4. Estimate the resistance of the new hull with Holtrop–Mennen, and know how far to trust it.

The parent is the KRISO Container Ship (KCS), Tokyo 2015 CFD workshop (NMRI): see the Bibliography, *Hull geometry*.

Sources: Papanikolaou (2026) §4.3 (linear distortion, variation of the parallel body, Lackenby method, Eqs. 4.10–4.19); Lackenby (1950); Holtrop and Mennen (1982), as in Papanikolaou (2026) §2.14; Extended reading *Hull Form Variation from a Parent*.

**How to run:** in Google Colab, first *File › Save a copy in Drive* (so your work is kept), then run the cells from the top with Shift+Enter. Change only the lines marked `# INPUT`.

## 1 · The parent: KCS offsets
`KCS_offsets.csv`: one row per station (x from the AP, positive forward, metres), one column per waterline (z every 0.1 m above the baseline), each value the half-breadth y in metres; 0 means no hull there. Full scale, cut from the KCS IGES file.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
URL = "https://raw.githubusercontent.com/L-u-i-s-B-a-t-i-s-t-a/PNSO-2026/main/sources/kcs_hull/KCS_offsets.csv"
try:
    raw = np.genfromtxt("KCS_offsets.csv", delimiter=",", skip_header=1)   # local copy, if you have one
except OSError:
    raw = np.genfromtxt(URL, delimiter=",", skip_header=1)                   # otherwise from the course site
X0 = raw[:, 0]                       # station positions [m]
Y0 = raw[:, 1:]                      # half-breadths [m], shape (stations, waterlines)
Z0 = np.round(np.arange(Y0.shape[1]) * 0.1, 1)   # waterline heights [m]
L0, B0, T0 = 230.0, 32.2, 10.8       # KCS: Lpp, B, T (workshop page)
print(f"{len(X0)} stations from x = {X0.min():.1f} to {X0.max():.1f} m; {len(Z0)} waterlines from 0 to {Z0.max():.1f} m")

## 2 · Hydrostatics from the offsets
Section area A(x) = 2∫y dz up to T; volume ∇ = ∫A dx; LCB from the first moment of A(x). Then C_B = ∇/(L·B·T), C_M = A_M/(B·T), C_P = ∇/(A_M·L), C_WP = A_WP/(L·B).
These are pure geometry: the same function works for the parent and for every variant.

In [ ]:
def hydrostatics(x, y, z, L, B, T):
    """x stations [m], y half-breadths (stations, waterlines) [m], z waterlines [m]; L = Lpp, x = 0 at the AP."""
    m = z <= T + 1e-9
    A = 2 * np.trapezoid(y[:, m], z[m], axis=1)              # section areas [m2]
    V = np.trapezoid(A, x)                                    # displaced volume [m3]
    xB = np.trapezoid(A * x, x) / V                           # centre of buoyancy from the AP [m]
    AM = np.interp(L / 2, x, A)                               # midship section area [m2]
    jT = np.argmin(abs(z - T))
    AWP = 2 * np.trapezoid(y[:, jT], x)                       # waterplane area [m2]
    wet = x[y[:, jT] > 0]
    iF = np.argmin(abs(x - L))                                # bulb section at the FP
    ABT = A[iF]
    hB = np.trapezoid(2 * y[iF, m] * z[m], z[m]) / ABT if ABT > 0 else 0.0
    return dict(V=V, CB=V / (L * B * T), CM=AM / (B * T), CP=V / (AM * L), CWP=AWP / (L * B),
                LCB=(xB - L / 2) / L * 100, LWL=wet.max() - wet.min(), ABT=ABT, hB=hB, A=A)

h0 = hydrostatics(X0, Y0, Z0, L0, B0, T0)
print("            offsets   published")
print(f"V [m3]      {h0['V']:8,.0f}   52,030")
print(f"C_B         {h0['CB']:8.4f}   0.6505")
print(f"C_M         {h0['CM']:8.4f}")
print(f"C_P         {h0['CP']:8.4f}")
print(f"C_WP        {h0['CWP']:8.4f}")
print(f"LCB [% L]   {h0['LCB']:+8.2f}   (+ forward of midship)")
print(f"LWL [m]     {h0['LWL']:8.1f}")

## 3 · Lackenby: change C_P and LCB at the parent's size
Each half of the hull is treated separately. ξ is the distance of a station from midship as a fraction of the half-length (0 at midship, 1 at the FP or AP).

Every station moves away from midship by **δξ = c·(1 − ξ)**: midship moves by c, the end does not move, and the gap opened at midship is filled with the midship section (new parallel middle body). The section shapes are never redrawn.

The prismatic coefficient of that half becomes **C_P,new = c + (1 − c)·C_P**, so **c = δC_P / (1 − C_P)**. This is Lackenby's Eq. 4.17 in Papanikolaou (2026) with the new parallel body chosen as δL_P = c (the second term of 4.17 then vanishes).

Two unknowns (c_F, c_A) and two targets (C_P, LCB): a 2 × 2 solve. A negative c (a finer half) removes parallel body near midship; the KCS has very little, so finer hulls are limited.

In [ ]:
XM = L0 / 2                                                   # midship [m]

def half_cp(x, y):
    """Prismatic coefficient of the fore and aft halves (0 to L/2 and L/2 to L) at T0."""
    m = Z0 <= T0 + 1e-9
    A = 2 * np.trapezoid(y[:, m], Z0[m], axis=1); AM = np.interp(XM, x, A)
    f = (x >= XM) & (x <= L0); a = (x >= 0) & (x <= XM)
    return np.trapezoid(A[f], x[f]) / (AM * XM), np.trapezoid(A[a], x[a]) / (AM * XM)

def lackenby(cF, cA, nx=1221):
    """Shift the KCS stations by dxi = c (1 - xi) in each half and return a new offsets grid on regular stations."""
    xi = np.where(X0 >= XM, (X0 - XM) / (L0 - XM), (XM - X0) / XM)
    c = np.where(X0 >= XM, cF, cA)
    xc = np.clip(xi, 0, 1)
    new = np.where(c >= 0, xc + c * (1 - xc), (xc + c) / (1 + c))   # c < 0: stations move in, those inside midship drop
    new = np.where(xi > 1, xi, new)                                  # beyond the AP and FP: no shift
    xn = np.where(X0 >= XM, XM + new * (L0 - XM), XM - new * XM)
    keep = ~(((X0 < XM) & (xn > XM)) | ((X0 >= XM) & (xn < XM)))
    x = np.linspace(X0.min(), X0.max(), nx)
    y = np.zeros((nx, len(Z0)))
    for j in range(len(Z0)):
        o = np.argsort(xn[keep])
        y[:, j] = np.interp(x, xn[keep][o], Y0[keep, j][o])
        ym = np.interp(XM, X0, Y0[:, j])                         # midship section fills the new parallel body
        y[(x >= XM - max(cA, 0) * XM) & (x <= XM + max(cF, 0) * (L0 - XM)), j] = ym
    return x, y

def solve_lackenby(CB_target, LCB_target):
    """Find cF, cA that give the target C_B and LCB (% L, + forward) on the parent."""
    p = np.zeros(2)
    for _ in range(15):
        h = hydrostatics(*lackenby(*p), Z0, L0, B0, T0)
        r = np.array([h["CB"] - CB_target, h["LCB"] - LCB_target])
        if abs(r[0]) < 2e-4 and abs(r[1]) < 0.02:
            return p, h
        J = np.zeros((2, 2)); e = 2e-3
        for k in range(2):
            q = p.copy(); q[k] += e
            hk = hydrostatics(*lackenby(*q), Z0, L0, B0, T0)
            J[:, k] = [(hk["CB"] - h["CB"]) / e, (hk["LCB"] - h["LCB"]) / e]
        p = p - np.linalg.solve(J, r)
    return p, h

CPF0, CPA0 = half_cp(X0, Y0)
print(f"parent: forebody C_P {CPF0:.3f}, aftbody C_P {CPA0:.3f}, whole C_P {h0['CP']:.3f}, LCB {h0['LCB']:+.2f} %")

In [ ]:
CB_target = 0.690      # INPUT: block coefficient of the variant (about 0.64 to 0.71)
LCB_target = -1.00     # INPUT: LCB of the variant, % L, + forward of midship

(cF, cA), h1 = solve_lackenby(CB_target, LCB_target)
xL, yL = lackenby(cF, cA)
CPF1, CPA1 = half_cp(xL, yL)
print(f"Lackenby factors: c_F = {cF:+.3f}, c_A = {cA:+.3f}")
print(f"forebody C_P {CPF0:.3f} -> {CPF1:.3f}   (check: c + (1 - c) C_P = {cF + (1 - cF) * CPF0:.3f})")
print(f"aftbody  C_P {CPA0:.3f} -> {CPA1:.3f}   (check: {cA + (1 - cA) * CPA0:.3f})")
print(f"C_B {h0['CB']:.3f} -> {h1['CB']:.3f}, C_P {h0['CP']:.3f} -> {h1['CP']:.3f}, C_M {h0['CM']:.3f} -> {h1['CM']:.3f}, LCB {h0['LCB']:+.2f} -> {h1['LCB']:+.2f} %")
print(f"new parallel body: {cF * XM:.1f} m forward + {cA * XM:.1f} m aft of midship = {(cF + cA) * XM / L0 * 100:.0f} % of L")

## 4 · Stretch to new L, B, T and D
Below the waterline every offset point moves as (x, y, z) → (s_x·x, s_y·y, s_z·z), with s_x = L/L₀, s_y = B/B₀, s_z = T/T₀.
Volume scales with s_x·s_y·s_z, exactly like L·B·T, so **C_B, C_M, C_P, C_WP and LCB (% L) do not change**; only the size does. This is the linear (affine) distortion of Papanikolaou (2026) §4.3.

**Depth.** Scaling everything by s_z would put the side at 17.2 m × T/T₀, not at the depth D of your design (from the container grid). So the part above the design waterline is stretched on its own, so that the side height of the KCS offsets, D₀ = 17.2 m at midship, lands on D:

z′ = s_z·z for z ≤ T₀,  z′ = T + (z − T₀)·(D − T)/(D₀ − T₀) for z > T₀.

Nothing below the waterline changes, so the hydrostatics at T are the same; the hull above T, which the bay-by-bay container count needs, reaches D. The bow and the stern above the side height (the KCS bow rises to 23.5 m) are stretched by the same factor.

In [ ]:
L = 200.0     # INPUT: Lpp [m]  (example ship, not a project value)
B = 30.0      # INPUT: B [m]
T = 10.5      # INPUT: T [m]
D = 16.0      # INPUT: depth to the main deck at side [m]
D0 = 17.2     # side height of the KCS offsets at midship [m] (top of the surfaces)

def stretch(x, y, L, B, T, D):
    """Scale a hull given on the KCS grid (x [m], y [m] on the waterlines Z0) to L, B, T and depth D."""
    sx, sy, sz = L / L0, B / B0, T / T0
    z = np.where(Z0 <= T0, Z0 * sz, T + (Z0 - T0) * (D - T) / (D0 - T0))
    return x * sx, y * sy, z

sx, sy, sz = L / L0, B / B0, T / T0
x1, y1, z1 = stretch(xL, yL, L, B, T, D)
h2 = hydrostatics(x1, y1, z1, L, B, T)
print(f"scale factors: s_x {sx:.3f}, s_y {sy:.3f}, s_z {sz:.3f} below T, {(D - T)/(D0 - T0):.3f} above T")
print(f"L/B {L0/B0:.2f} -> {L/B:.2f}, B/T {B0/T0:.2f} -> {B/T:.2f}, D/T {D0/T0:.2f} -> {D/T:.2f}")
k = np.argmin(abs(x1 - L / 2)); top = np.where(y1[k] > 0)[0].max()
print(f"side height at midship: {z1[top]:.2f} m (D = {D:.2f} m)")
print("            after Lackenby   after stretch")
for k_, f in [("CB", "{:.4f}"), ("CM", "{:.4f}"), ("CP", "{:.4f}"), ("CWP", "{:.4f}"), ("LCB", "{:+.2f}")]:
    print(f"{k_:11s} {f.format(h1[k_]):>14s}   {f.format(h2[k_]):>13s}")
print(f"V [m3]      {h1['V']:14,.0f}   {h2['V']:13,.0f}   (ratio {h2['V']/h1['V']:.4f} = s_x s_y s_z {sx*sy*sz:.4f})")

## 5 · Look at the result

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
A0 = h0["A"] / (B0 * T0); A2 = h2["A"] / (B * T)
ax[0].plot(X0 / L0, A0, label=f"KCS parent, C_B {h0['CB']:.3f}")
ax[0].plot(x1 / L, A2, "--", label=f"variant, C_B {h2['CB']:.3f}")
ax[0].set_xlabel("x / L from the AP"); ax[0].set_ylabel("A(x) / (B·T)"); ax[0].set_title("Sectional area curves (to T)")
ax[0].grid(alpha=0.3); ax[0].legend()

for xs, ys, zs, LL, BB, TT, st, lab in [(X0, Y0, Z0, L0, B0, T0, "-", "KCS"), (x1, y1, z1, L, B, T, "--", "variant")]:
    for f in np.linspace(0, 1, 21):
        k = np.argmin(abs(xs - f * LL)); yy = ys[k, Z0 <= 17.2] / (BB / 2); zz = zs[Z0 <= 17.2] / TT
        ok = yy > 0.002
        sgn = 1 if f >= 0.5 else -1
        ax[1].plot(sgn * np.where(ok, yy, np.nan), zz, st, color="C0" if lab == "KCS" else "C1", lw=0.9)
ax[1].axhline(1, color="grey", ls=":"); ax[1].set_aspect("equal")
ax[1].set_xlabel("y / (B/2)   (aft left, fore right)"); ax[1].set_ylabel("z / T"); ax[1].set_title("Body plan, non-dimensional: KCS solid, variant dashed")
plt.tight_layout(); plt.show()

## 6 · Resistance with Holtrop–Mennen
The hull form inputs (C_M, C_WP, LCB, waterline length, bulb area and height) are taken **from the hull**, not from regressions. The function is the course one (H2, Tool 2.4).

In [ ]:
import math
RHO, G, KN, NU = 1.025, 9.81, 0.514444, 1.1883e-6
def holtrop_mennen(V_kn, L, B, T, vol, lcb=0.0, CM=0.98, CWP=None, ABT=0.0, hB=0.0, AT=0.0,
                   SAPP=0.0, k2=1.5, Cstern=0, Dp=None, twin=False, AE_A0=0.6, PD=1.0, rho=RHO):
    """Holtrop and Mennen (1982), Int. Shipbuilding Progress 29(335); Papanikolaou (2026) §2.14, footnote 34.
    L is the waterline length. Same formulas as Tool 2.4. Returns resistances in kN and power in kW."""
    RHO_ = 1000 * rho; TF = TA = T
    Dp = Dp or 0.65 * T
    V = V_kn * KN
    CB = vol / (L * B * T); CP = CB / CM
    CWP = CWP or (1 + 2 * CB) / 3
    Fn = V / math.sqrt(G * L)
    S = L * (2 * T + B) * math.sqrt(CM) * (0.453 + 0.4425 * CB - 0.2862 * CM - 0.003467 * B / T + 0.3696 * CWP) + 2.38 * ABT / CB
    Rn = V * L / NU; CF = 0.075 / (math.log10(Rn) - 2) ** 2
    LR = L * (1 - CP + 0.06 * CP * lcb / (4 * CP - 1))
    TL = T / L
    c12 = TL ** 0.2228446 if TL > 0.05 else (48.20 * (TL - 0.02) ** 2.078 + 0.479948 if TL > 0.02 else 0.479948)
    c13 = 1 + 0.003 * Cstern
    k1 = c13 * (0.93 + c12 * (B / LR) ** 0.92497 * (0.95 - CP) ** -0.521448 * (1 - CP + 0.0225 * lcb) ** 0.6906)
    q = 0.5 * RHO_ * V * V
    RF = q * S * CF; RAPP = q * SAPP * k2 * CF
    BL = B / L
    c7 = 0.229577 * BL ** 0.33333 if BL < 0.11 else (BL if BL < 0.25 else 0.5 - 0.0625 / BL)
    iE = 1 + 89 * math.exp(-(L / B) ** 0.80856 * (1 - CWP) ** 0.30484 * (1 - CP - 0.0225 * lcb) ** 0.6367
                           * (LR / B) ** 0.34574 * (100 * vol / L ** 3) ** 0.16302)
    c1 = 2223105 * c7 ** 3.78613 * (T / B) ** 1.07961 * (90 - iE) ** -1.37565
    c3 = 0.56 * ABT ** 1.5 / (B * T * (0.31 * math.sqrt(ABT) + TF - hB)) if ABT > 0 else 0.0
    c2 = math.exp(-1.89 * math.sqrt(c3))
    c5 = 1 - 0.8 * AT / (B * T * CM)
    lam = 1.446 * CP - 0.03 * L / B if L / B < 12 else 1.446 * CP - 0.36
    c16 = 8.07981 * CP - 13.8673 * CP ** 2 + 6.984388 * CP ** 3 if CP < 0.8 else 1.73014 - 0.7067 * CP
    m1 = 0.0140407 * L / T - 1.75254 * vol ** (1 / 3) / L - 4.79323 * B / L - c16
    L3V = L ** 3 / vol
    c15 = -1.69385 if L3V < 512 else (-1.69385 + (L / vol ** (1 / 3) - 8.0) / 2.36 if L3V < 1727 else 0.0)
    m2 = c15 * CP ** 2 * math.exp(-0.1 / Fn ** 2)
    RW = c1 * c2 * c5 * vol * RHO_ * G * math.exp(m1 * Fn ** -0.9 + m2 * math.cos(lam / Fn ** 2))
    RB = 0.0
    if ABT > 0:
        PBb = 0.56 * math.sqrt(ABT) / (TF - 1.5 * hB)
        Fni = V / math.sqrt(G * (TF - hB - 0.25 * math.sqrt(ABT)) + 0.15 * V * V)
        RB = 0.11 * math.exp(-3 * PBb ** -2) * Fni ** 3 * ABT ** 1.5 * RHO_ * G / (1 + Fni ** 2)
    RTR = 0.0
    if AT > 0:
        FnT = V / math.sqrt(2 * G * AT / (B + B * CWP))
        RTR = q * AT * 0.2 * (1 - 0.2 * FnT) if FnT < 5 else 0.0
    c4 = TF / L if TF / L <= 0.04 else 0.04
    CA = 0.006 * (L + 100) ** -0.16 - 0.00205 + 0.003 * math.sqrt(L / 7.5) * CB ** 4 * c2 * (0.04 - c4)
    RA = q * S * CA
    RT = RF * k1 + RAPP + RW + RB + RTR + RA
    CV = (k1 + (k2 - k1) * SAPP / (S + SAPP)) * CF + CA
    CP1 = 1.45 * CP - 0.315 - 0.0225 * lcb
    if twin:
        w = 0.3095 * CB + 10 * CV * CB - 0.23 * Dp / math.sqrt(B * T)
        t = 0.325 * CB - 0.1885 * Dp / math.sqrt(B * T)
        etaR = 0.9737 + 0.111 * (CP - 0.0225 * lcb) - 0.06325 * PD
    else:
        BTA = B / TA
        c8 = B * S / (L * Dp * TA) if BTA < 5 else S * (7 * BTA - 25) / (L * Dp * (BTA - 3))
        c9 = c8 if c8 < 28 else 32 - 16 / (c8 - 24)
        c11 = TA / Dp if TA / Dp < 2 else 0.0833333 * (TA / Dp) ** 3 + 1.33333
        w = (c9 * CV * L / TA * (0.0661875 + 1.21756 * c11 * CV / (1 - CP1)) + 0.24558 * math.sqrt(B / (L * (1 - CP1)))
             - 0.09726 / (0.95 - CP) + 0.11434 / (0.95 - CB) + 0.75 * Cstern * CV + 0.002 * Cstern)
        c10 = B / L if L / B > 5.2 else 0.25 - 0.003328402 / (B / L - 0.134615385)
        t = 0.001979 * L / (B - B * CP1) + 1.0585 * c10 - 0.00524 - 0.1418 * Dp ** 2 / (B * T) + 0.0015 * Cstern
        etaR = 0.9922 - 0.05908 * AE_A0 + 0.07424 * (CP - 0.0225 * lcb)
    return dict(Fn=Fn, CB=CB, CP=CP, CWP=CWP, S=S, k1=k1, RFk=RF * k1 / 1e3, RW=RW / 1e3, RA=RA / 1e3,
                RAPP=RAPP / 1e3, RB=RB / 1e3, RTR=RTR / 1e3, RT=RT / 1e3, PE=RT * V / 1e3,
                w=w, t=t, etaR=etaR, etaH=(1 - t) / (1 - w))

In [ ]:
V_kn = 20.0      # INPUT: speed [kn]

def resistance(h, B, T, V_kn):
    return holtrop_mennen(V_kn, h["LWL"], B, T, h["V"], lcb=h["LCB"], CM=h["CM"], CWP=h["CWP"],
                          ABT=h["ABT"], hB=h["hB"], Cstern=0)

r = resistance(h2, B, T, V_kn)
print(f"variant: Fn {r['Fn']:.3f}, R_T {r['RT']:,.0f} kN (friction with form factor {r['RFk']:,.0f}, wave {r['RW']:,.0f}, bulb {r['RB']:,.1f}, correlation {r['RA']:,.0f}), P_E {r['PE']:,.0f} kW")

**How far to trust it.** On the KCS itself, at model scale (L = 7.2786 m, Fr = 0.26, Re = 1.46 × 10⁷), the measured total resistance coefficient is C_T = 3.711 × 10⁻³ (towing-tank data as quoted in MDPI *Mathematics* 13(11):1788, 2025; the model had its rudder and was free to sink and trim). The cell below makes the same prediction with Holtrop–Mennen.

In [ ]:
Lm = 7.2786; s = L0 / Lm; Vm = 0.26 * np.sqrt(9.81 * Lm)
rm = holtrop_mennen(Vm / KN, h0["LWL"] / s, B0 / s, T0 / s, h0["V"] / s**3, lcb=h0["LCB"], CM=h0["CM"], CWP=h0["CWP"],
                    ABT=h0["ABT"] / s**2, hB=h0["hB"] / s, Cstern=0)
q = 0.5 * 1025 * rm["S"] * Vm**2
CF = 0.075 / (np.log10(1.46e7) - 2) ** 2                        # ITTC-57 at the test Reynolds number
CT = rm["k1"] * CF + (rm["RW"] + rm["RB"] + rm["RTR"]) * 1e3 / q  # no correlation allowance at model scale
print(f"Holtrop-Mennen C_T = {CT*1e3:.3f}e-3 vs measured 3.711e-3: {(CT/3.711e-3-1)*100:+.1f} %")

## 7 · A family of hulls
The same L, B, T and speed, with C_B from the parent's value to 0.71. This is the kind of table an optimization model needs: the hull is now a function of (C_B, LCB, L, B, T).

In [ ]:
rows = []
for cb in np.arange(0.655, 0.7151, 0.015):          # INPUT: range of C_B
    (cf, ca), _ = solve_lackenby(cb, LCB_target)
    xs, ys = lackenby(cf, ca)
    hh = hydrostatics(*stretch(xs, ys, L, B, T, D), L, B, T)
    rr = resistance(hh, B, T, V_kn)
    rows.append((hh["CB"], hh["LCB"], cf, ca, hh["V"], rr["RT"], rr["PE"]))
print(" C_B    LCB %    c_F     c_A     V [m3]   R_T [kN]  P_E [kW]")
for row in rows:
    print(f"{row[0]:.3f}  {row[1]:+.2f}  {row[2]:+.3f}  {row[3]:+.3f}  {row[4]:8,.0f}  {row[5]:7,.0f}  {row[6]:8,.0f}")
rows = np.array(rows)
plt.figure(figsize=(6, 4)); plt.plot(rows[:, 0], rows[:, 6], "o-")
plt.xlabel("C_B"); plt.ylabel("P_E [kW]"); plt.title(f"Effective power at {V_kn:.0f} kn, L {L:.0f} m, B {B:.0f} m, T {T:.1f} m"); plt.grid(alpha=0.3); plt.show()

## 8 · Limits: state them in your report
- **Fullness:** Lackenby fuller is clean up to about C_B 0.71 (δC_P ≈ 0.06, about 17 % of L of new parallel body). Finer than the parent removes parallel body, and the KCS has very little: below about C_B 0.64 the result is crude.
- **Proportions:** keep L/B within about 5.5–7.5 and B/T within about 2.5–3.5 (the KCS has 7.14 and 2.98). Further away, the stretched bulb, stern and sections stop being realistic. These ranges are a course guideline, not a rule from a source.
- **Bulb and stern** are stretched with the hull, not redesigned for the new length and speed.
- **Resistance** comes from Holtrop–Mennen with the hull's own coefficients, not from model tests: about 6 % above the model test on the parent. Use it to compare variants with each other; quote a margin on any absolute power.
- **Cite** the KCS geometry (Tokyo 2015 workshop, NMRI) and Lackenby (1950).